# C-Space Slice Explorer

Standalone, minimal-setup notebook for exploring the grasp/placement/c-free
polytopes' 2-D cross-sections interactively (`cspace_slice_visualizer.py`),
independent of `full_arm_planner.ipynb`'s own kernel.

**Run this in its own kernel** so it stays responsive while a planner run is
mid-solve elsewhere - a running cell blocks its own kernel's widget callbacks
entirely, so this can't live inside `full_arm_planner.ipynb`'s kernel for the
"explore in the meantime" use case.

Section 3 gets you a working slice viewer with no Meshcat/3-D view. Section 4
is optional - adds a live 3-D robot pose synced to the sliders, at the cost
of a heavier (`CspaceFreePolytope`-based) setup step.

In [12]:
import sys
from pathlib import Path

def _find_repo_root(start: Path, markers=(".git", "README.md")) -> Path:
    for candidate in (start, *start.parents):
        if any((candidate / m).exists() for m in markers):
            return candidate
    raise RuntimeError(f"Could not locate repo root above {start}")

REPO_ROOT = _find_repo_root(Path.cwd())
for _p in (REPO_ROOT, REPO_ROOT / "algorithms", REPO_ROOT / "algorithms" / "nlp_sampling"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import numpy as np
from pydrake.all import (
    RobotDiagramBuilder, Parser, RigidTransform, RollPitchYaw, RotationMatrix,
)
from pydrake.geometry.optimization import HPolyhedron, LoadIrisRegionsYamlFile

from cspace_slice_visualizer import CSpaceSliceVisualizer


## 1. Build the plant (obstacle-free, no Meshcat)

Same URDFs, same weld order as `full_arm_planner.ipynb`'s own plant (cells
3-5 there) - `my_sdfs/panda_arm.urdf` (the repo's custom-narrowed-wrist
variant), `panda_hand.urdf`, `bottle_cap.sdf` - minus the obstacles (not
needed for polytope-slice math) and minus Meshcat (see section 4 if you want
it). This keeps position indexing identical to `full_arm_planner.ipynb`'s
plant, which the index-parity assertions below double-check.

In [13]:
builder = RobotDiagramBuilder(time_step=0.0)
plant = builder.plant()
scene_graph = builder.scene_graph()
parser = Parser(plant, scene_graph)
parser.SetAutoRenaming(True)

panda_arm = parser.AddModels(url="file://" + str(REPO_ROOT / "my_sdfs" / "panda_arm.urdf"))[0]
panda_hand = parser.AddModels(url="package://drake_models/franka_description/urdf/panda_hand.urdf")[0]

plant.WeldFrames(
    plant.world_frame(),
    plant.GetFrameByName("panda_link0", panda_arm),
    RigidTransform())

X_8H = RigidTransform(RollPitchYaw(0.0, 0.0, -np.deg2rad(45.0)), [0.0, 0.0, 0.0])
plant.WeldFrames(
    plant.GetFrameByName("panda_link8", panda_arm),
    plant.GetFrameByName("panda_hand", panda_hand),
    X_8H)
plant.GetJointByName("panda_finger_joint1", panda_hand).set_default_translation(-0.024)

cap = parser.AddModels(file_name=str(REPO_ROOT / "my_sdfs" / "bottle_cap.sdf"))[0]
plant.WeldFrames(
    plant.world_frame(),
    plant.GetFrameByName("base_link", cap),
    RigidTransform(RotationMatrix(), [0.5, 0, 0]))

plant.Finalize()
diagram = builder.Build()
diagram_context = diagram.CreateDefaultContext()
plant_context = plant.GetMyMutableContextFromRoot(diagram_context)

print(f"num_positions = {plant.num_positions()}")
for i, name in enumerate(plant.GetPositionNames()):
    print(f"{i:2d}: {name}")


num_positions = 10
 0: panda_panda_joint1_q
 1: panda_panda_joint2_q
 2: panda_panda_joint3_q
 3: panda_panda_joint4_q
 4: panda_panda_joint5_q
 5: panda_panda_joint6_q
 6: panda_panda_joint7_q
 7: panda_hand_panda_finger_joint1_x
 8: panda_hand_panda_finger_joint2_x
 9: bottle_cap_cap_to_base_q


## 2. Grasp/placement polytopes from the wrist-axis analysis (#80)

Reuses `wrist_axis_grasp_polytope.py` exactly as `full_arm_planner.ipynb`
cells 10/12 do (see `tc_space_investigation_summary.md` Part 2 for the
derivation). `build_constraints()` builds its own separate, obstacle-free,
AutoDiff-augmented plant purely to compute the grasp seed `q0` and the
wrist-axis margin box - the index-parity asserts confirm that plant and this
notebook's plant above agree on position ordering before reusing arrays
between them.

In [14]:
from wrist_axis_grasp_polytope import (
    ARM_JOINT_INDICES, build_constraints, find_grasp_seed, build_simple_margin_polytope,
)
from nlp_sampling_standalone_test import make_joint_indices

GRASP_MARGIN = 0.001  # matches the finger joints' own grasp tolerance width -
                       # see tc_space_investigation_summary.md Part 2.

constraints = build_constraints()
q0 = find_grasp_seed(constraints)
lower_wrist_box, upper_wrist_box = build_simple_margin_polytope(constraints, q0, GRASP_MARGIN)

idx = make_joint_indices(plant, panda_arm, panda_hand, cap)
idx_wrist, idx_f1, idx_f2, idx_cap = idx["idx_wrist"], idx["idx_f1"], idx["idx_f2"], idx["idx_cap"]

assert list(idx["idx_arm"][:6]) == ARM_JOINT_INDICES
assert idx_wrist == constraints["idx_wrist"] == 6
assert (idx_f1, idx_f2, idx_cap) == (7, 8, 9)
print("index-parity assertions OK")
print(f"Grasp seed q0 = {np.round(q0, 4)}")

grasp_lower, grasp_upper = lower_wrist_box.copy(), upper_wrist_box.copy()
grasp_lower[idx_f1], grasp_upper[idx_f1] = -0.025, -0.024
grasp_lower[idx_f2], grasp_upper[idx_f2] = 0.024, 0.025
grasp_lower[idx_cap] = plant.GetPositionLowerLimits()[idx_cap]
grasp_upper[idx_cap] = plant.GetPositionUpperLimits()[idx_cap]
grasp_polytope = HPolyhedron.MakeBox(grasp_lower, grasp_upper)

placement_lower = plant.GetPositionLowerLimits().copy()
placement_upper = plant.GetPositionUpperLimits().copy()
placement_lower[idx_wrist], placement_upper[idx_wrist] = -1.0, 1.0
placement_lower[idx_f1], placement_upper[idx_f1] = -0.055, -0.024
placement_lower[idx_f2], placement_upper[idx_f2] = 0.024, 0.055
placement_polytope = HPolyhedron.MakeBox(placement_lower, placement_upper)

print("q0 in placement:", placement_polytope.PointInSet(q0), " q0 in grasp:", grasp_polytope.PointInSet(q0))


index-parity assertions OK
Grasp seed q0 = [-1.7375 -1.6525  1.8095 -2.2268  1.7824  1.708   0.9944 -0.024   0.0247
  0.9449]
q0 in placement: True  q0 in grasp: True


## 3. C-free regions from disk, then launch the slice viewer

No IRIS-ZO/clique-cover solve needed here - loads whatever's already been
computed and saved (`data/cfree/cfree_full_98coverage.yaml`, or point this at
a fresher file `full_arm_planner.ipynb` has since written, e.g.
`cfree_99coverage_irisnp.yaml`).

In [17]:
# cfree_path = REPO_ROOT / "data" / "cfree" / "cfree_97coverage_iriszo(64 regions).yaml"
# cfree_regions = list(LoadIrisRegionsYamlFile(str(cfree_path)).values())

cfree_path = REPO_ROOT / "data" / "cfree" / "cfree_10dof_irisnp2_98_k6.yaml"
cfree_regions = list(LoadIrisRegionsYamlFile(str(cfree_path)).values())

print(f"Loaded {len(cfree_regions)} c-free regions.")


Loaded 6 c-free regions.


In [18]:
# slice_viz = CSpaceSliceVisualizer(
#     plant, plant_context,
#     groups={
#         "c-free": (cfree_regions, "steelblue"),
#         "grasp": (grasp_polytope, "seagreen"),
#         "placement": (placement_polytope, "darkorange"),
#     },
#     q0=q0,
#     free_axes=(idx_f1, idx_f2),  # the two DOF the grasp polytope leaves free
# )
# slice_viz.show()


## 4. Live 3-D robot view via Meshcat

Rebuilds a second, Meshcat-visualized plant (same URDFs/weld order/obstacles
as `full_arm_planner.ipynb` cells 3-6) via `CIrisPlantVisualizer`, then
rebinds a new slice viewer onto it with `CSpaceSliceVisualizer.from_visualizer`
so dragging any slider also moves a visible robot pose. Like
`full_arm_planner.ipynb`, it skips the optional `CspaceFreePolytope`, so it
builds quickly.

In [19]:
from pydrake.all import Role
from ciris_plant_visualizer import CIrisPlantVisualizer

builder_viz = RobotDiagramBuilder(time_step=0.0)
plant_viz = builder_viz.plant()
scene_graph_viz = builder_viz.scene_graph()
parser_viz = Parser(plant_viz, scene_graph_viz)
parser_viz.SetAutoRenaming(True)

panda_arm_viz = parser_viz.AddModels(url="file://" + str(REPO_ROOT / "my_sdfs" / "panda_arm.urdf"))[0]
panda_hand_viz = parser_viz.AddModels(url="package://drake_models/franka_description/urdf/panda_hand.urdf")[0]
plant_viz.WeldFrames(plant_viz.world_frame(), plant_viz.GetFrameByName("panda_link0", panda_arm_viz), RigidTransform())
plant_viz.WeldFrames(
    plant_viz.GetFrameByName("panda_link8", panda_arm_viz),
    plant_viz.GetFrameByName("panda_hand", panda_hand_viz),
    RigidTransform(RollPitchYaw(0.0, 0.0, -np.deg2rad(45.0)), [0.0, 0.0, 0.0]))
plant_viz.GetJointByName("panda_finger_joint1", panda_hand_viz).set_default_translation(-0.024)

cap_viz = parser_viz.AddModels(file_name=str(REPO_ROOT / "my_sdfs" / "bottle_cap.sdf"))[0]
obstacle1_viz = parser_viz.AddModels(file_name=str(REPO_ROOT / "my_sdfs" / "obstacle.sdf"))[0]
obstacle3_viz = parser_viz.AddModels(file_name=str(REPO_ROOT / "my_sdfs" / "obstacle.sdf"))[0]
plant_viz.WeldFrames(plant_viz.world_frame(), plant_viz.GetFrameByName("base_link", cap_viz),
                     RigidTransform(RotationMatrix(), [0.5, 0, 0]))
plant_viz.WeldFrames(plant_viz.world_frame(), plant_viz.GetFrameByName("obstacle_link", obstacle1_viz),
                     RigidTransform(RotationMatrix(), [0.51, 0.031, 0.01]))
plant_viz.WeldFrames(plant_viz.world_frame(), plant_viz.GetFrameByName("obstacle_link", obstacle3_viz),
                     RigidTransform(RotationMatrix(), [0.467, -0.005, 0.01]))
plant_viz.Finalize()

# No CspaceFreePolytope - same as full_arm_planner.ipynb, it's optional and slow to build.
visualizer = CIrisPlantVisualizer(
    plant_viz, builder_viz, scene_graph_viz,
    viz_role=Role.kIllustration, allow_plus_3dof=True)
visualizer.task_space_diagram.ForcedPublish(visualizer.task_space_diagram_context)

INFO:drake:Meshcat listening for connections at http://localhost:7004


Visualisations won't work properly. Can't visualize the TC-Space of plants with more than 3-DOF. The first 3 DOF from the plant will be visualized


In [11]:
slice_viz_3d = CSpaceSliceVisualizer.from_visualizer(
    visualizer,
    groups={
        "c-free": (cfree_regions, "steelblue"),
        "grasp": (grasp_polytope, "seagreen"),
        # "placement": (placement_polytope, "darkorange"),
    },
    q0=q0,
    free_axes=(idx_wrist, idx_cap),
)
slice_viz_3d.show()

## 5. Once `full_arm_planner.ipynb` has a trajectory

That notebook's own kernel needs to be free (its solve cell finished) before
you can run its integration cell there directly (`slice_viz.set_trajectory(traj)`).
If you'd rather see it here without switching kernels, save the sampled path
from that notebook (`np.save(REPO_ROOT / "artifacts" / "traj_path.npy", np.array(traj_path))`)
and load it in this one:

```python
traj_path = np.load(REPO_ROOT / "artifacts" / "traj_path.npy")
slice_viz.set_path(list(traj_path))
```